# 01 — Leave-One-Trajectory-Out Folds + Hurdle-Aware Targets

**Goal of this notebook:** turn the *already generated* synchronized multimodal
KFUPM dataset (`synchronized_multimodal_five_gnb_dataset.npz`, produced by the
original `KFUPM_Agentic_WCNC_Improved.ipynb`) into a more rigorous, more
learnable set of forecasting sequences — **without re-running Sionna ray
tracing, the virtual-LiDAR extraction, or any GPU scene simulation.**

This notebook only reads the dataset that already exists under
`improved_experiment/data/`. All outputs go to a **new** sibling folder,
`improved_experiment_v2/`, so nothing here overwrites or mixes with the
original WCNC results.

### Why this step is needed (diagnosis from the original run)

Looking at the original notebook's own printed metrics
(`forecaster_test_metrics.json`, `agentic_handover_metrics.json`):

* MAE for every forecaster is **~20 dB**, RMSE **29–38 dB** — far higher than
  a well-posed RSRP-forecasting task should need. The cause is visible in the
  sequence-preparation cell: when a gNB has **no valid ray-traced path**
  (`valid_path_counts == 0`), its power is set to a sentinel floor
  (`-300 dB`, clipped to `-220 dB`) and then trained/evaluated **as if it
  were a real regression target**. Because ~20–50% of gNB-timestep pairs
  have no path at all (see the per-gNB `valid_path_fraction` column in
  `rf_dense_five_gnb_summary.csv`), the model is effectively being scored on
  predicting an arbitrary constant for a large fraction of samples. This
  single artifact dominates the reported MAE/RMSE and masks the true
  forecasting quality of the proposed joint model.
* The train/val/test split uses **only one held-out trajectory each** for
  validation and testing (4/1/1 of 6 total). Every metric in the paper is
  therefore a **single-sample estimate** with no confidence interval — a
  reviewer at WCNC can reasonably ask "how much of this is just that one
  test trajectory?".
* The input feature vector is 97-D (15 RF + 10 mobility + **72 raw LiDAR
  rays**) trained on **892 windows**. The raw per-ray LiDAR block is very
  high-dimensional relative to the sample count and is a likely source of
  overfitting; the dataset already contains a compact 7-D
  `lidar_summary` block (nearest/mean/farthest/percentiles/hit-fraction)
  that we carry through as an ablation option.

### What this notebook produces
1. **Hurdle-aware targets**: the continuous power target is kept, but we also
   carry through the per-step, per-gNB **link-validity mask**
   (`valid_path_counts > 0`) so that downstream models can be trained and
   *fairly evaluated* without being penalized for guessing a sentinel value
   on non-existent links.
2. **Rotating leave-one-trajectory-out (LOTO) folds**: with 6 trajectories we
   build **6 folds**, each structured exactly like the original split
   (4 train / 1 validation / 1 test trajectories), just rotated so that
   *every* trajectory is used once as the test trajectory. This gives
   6 independent test estimates of every metric (mean ± std, paired
   significance tests) instead of 1.
3. Two parallel feature sets per fold: `lidar_mode="summary"` (7-D, the
   recommended default) and `lidar_mode="full"` (72-D, kept for a direct
   ablation in notebook 02).

Everything is saved under:
```
<PROJECT_ROOT>/improved_experiment_v2/data/fold_{k}_{lidar_mode}_sequences.npz
<PROJECT_ROOT>/improved_experiment_v2/results/loto_fold_manifest.json
```


In [1]:
from pathlib import Path
import json
import numpy as np

# ------------------------------------------------------------
# 1. Paths — reuse the ALREADY GENERATED dataset, write to a NEW folder
# ------------------------------------------------------------

PROJECT_ROOT = Path.home() / "wcnc_project"

OLD_DATA_DIR = PROJECT_ROOT / "improved_experiment" / "data"
MULTIMODAL_FILE = OLD_DATA_DIR / "synchronized_multimodal_five_gnb_dataset.npz"

V2_ROOT = PROJECT_ROOT / "improved_experiment_v2"
V2_DATA_DIR = V2_ROOT / "data"
V2_RESULTS_DIR = V2_ROOT / "results"

for folder in [V2_DATA_DIR, V2_RESULTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

if not MULTIMODAL_FILE.exists():
    raise FileNotFoundError(
        f"Expected the already-generated multimodal dataset at:\n{MULTIMODAL_FILE}\n"
        "This notebook does not regenerate it — run the original "
        "'Synchronized multimodal KFUPM dataset' cell first if it is missing."
    )

print("Reading existing dataset from:", MULTIMODAL_FILE)
print("New (non-mixing) output root: ", V2_ROOT)


Reading existing dataset from: /home/afanali/wcnc_project/improved_experiment/data/synchronized_multimodal_five_gnb_dataset.npz
New (non-mixing) output root:  /home/afanali/wcnc_project/improved_experiment_v2


In [2]:
# ------------------------------------------------------------
# 2. Load the already-generated synchronized multimodal dataset
# ------------------------------------------------------------

with np.load(MULTIMODAL_FILE, allow_pickle=True) as data:
    positions_xyz_m = np.asarray(data["positions_xyz_m"], dtype=np.float32)
    trajectory_ids = np.asarray(data["trajectory_ids"], dtype=np.int32)
    route_indices = np.asarray(data["route_indices"], dtype=np.int32)
    rf_power_db = np.asarray(data["rf_power_db"], dtype=np.float32)
    valid_path_counts = np.asarray(data["valid_path_counts"], dtype=np.float32)
    minimum_delay_s = np.asarray(data["minimum_delay_s"], dtype=np.float32)
    mobility_features = np.asarray(data["mobility_features"], dtype=np.float32)
    lidar_ranges_m = np.asarray(data["lidar_ranges_m"], dtype=np.float32)
    lidar_summary = np.asarray(data["lidar_summary"], dtype=np.float32)
    best_gnb_index = np.asarray(data["best_gnb_index"], dtype=np.int16)

    def decode_name(value):
        if isinstance(value, (bytes, np.bytes_)):
            return value.decode("utf-8", errors="replace")
        return str(value)

    gnb_names = [decode_name(name) for name in data["gnb_names"]]

number_of_observations = positions_xyz_m.shape[0]
number_of_gnbs = len(gnb_names)
unique_trajectory_ids = np.sort(np.unique(trajectory_ids))
number_of_trajectories = len(unique_trajectory_ids)

print("Observations:", number_of_observations)
print("gNBs:", gnb_names)
print("Trajectories:", unique_trajectory_ids.tolist())

if number_of_trajectories < 3:
    raise RuntimeError("At least 3 trajectories are required for a train/val/test split per fold.")


Observations: 1440
gNBs: ['gNB1_B59', 'gNB2_Building_44', 'gNB3_Building_54', 'gNB4_Building_29_Community_Center', 'gNB5_Building_617']
Trajectories: [0, 1, 2, 3, 4, 5]


In [3]:
# ------------------------------------------------------------
# 3. Forecasting configuration (unchanged from the original notebook,
#    so results stay directly comparable)
# ------------------------------------------------------------

TIME_STEP_S = 0.5
HISTORY_STEPS = 12
FORECAST_HORIZON = 6

POWER_FLOOR_DB = -220.0
POWER_CEILING_DB = 0.0

LIDAR_MIN_MODEL_RANGE_M = 0.5
LIDAR_MAX_MODEL_RANGE_M = 300.0

# Continuous power target (unchanged transform)
target_power_db = np.clip(rf_power_db, POWER_FLOOR_DB, POWER_CEILING_DB).astype(np.float32)

# NEW: explicit hurdle / link-availability target carried alongside the
# continuous target at every stage from here on, instead of being computed
# only after the fact for reporting.
target_valid = (valid_path_counts > 0).astype(np.float32)

model_rf_power_db = target_power_db.copy()
model_valid_paths = np.log1p(np.maximum(valid_path_counts, 0.0)).astype(np.float32)
model_delay_us = np.log1p(np.maximum(
    np.where(np.isfinite(minimum_delay_s) & (minimum_delay_s >= 0.0), minimum_delay_s * 1e6, 0.0),
    0.0,
)).astype(np.float32)
model_mobility = mobility_features.copy()
model_lidar_full = np.clip(lidar_ranges_m, LIDAR_MIN_MODEL_RANGE_M, LIDAR_MAX_MODEL_RANGE_M).astype(np.float32)
model_lidar_summary = lidar_summary.copy()

MOBILITY_NAMES = ["speed_mps", "velocity_x_mps", "velocity_y_mps", "acceleration_x_mps2",
                  "acceleration_y_mps2", "heading_sin", "heading_cos", "yaw_rate_radps",
                  "delta_x_m", "delta_y_m"]

def build_feature_block(lidar_mode):
    """lidar_mode: 'summary' (7-D, recommended) or 'full' (72-D, ablation)."""
    if lidar_mode == "summary":
        lidar_block = model_lidar_summary
        lidar_names = ["lidar_nearest_m", "lidar_mean_m", "lidar_farthest_m",
                       "lidar_p10_m", "lidar_p50_m", "lidar_p90_m", "lidar_hit_fraction"]
    elif lidar_mode == "full":
        lidar_block = model_lidar_full
        lidar_names = [f"lidar_range_{i:02d}_m" for i in range(lidar_block.shape[1])]
    else:
        raise ValueError(f"Unknown lidar_mode: {lidar_mode}")

    feats = np.concatenate(
        [model_rf_power_db, model_valid_paths, model_delay_us, model_mobility, lidar_block], axis=1
    ).astype(np.float32)

    names = (
        [f"{n}_power_db" for n in gnb_names]
        + [f"{n}_log_valid_paths" for n in gnb_names]
        + [f"{n}_log_delay_us" for n in gnb_names]
        + MOBILITY_NAMES
        + lidar_names
    )
    return feats, names

print("Feature block sizes:")
for mode in ["summary", "full"]:
    feats, names = build_feature_block(mode)
    print(f"  lidar_mode={mode:7s} -> input dimension {feats.shape[1]}")


Feature block sizes:
  lidar_mode=summary -> input dimension 32
  lidar_mode=full    -> input dimension 97


In [4]:
# ------------------------------------------------------------
# 4. Rotating leave-one-trajectory-out (LOTO) fold definition
# ------------------------------------------------------------

def rotating_fold_trajectories(fold_index, all_traj_ids):
    """
    Fold k: test = trajectory k, validation = trajectory k+1 (mod N),
    train = the remaining N-2 trajectories. This exactly reproduces the
    original notebook's 4-train / 1-val / 1-test structure, repeated N times
    so every trajectory is used once as the held-out test trajectory.
    """
    order = list(all_traj_ids)
    n = len(order)
    test_traj = [order[fold_index % n]]
    val_traj = [order[(fold_index + 1) % n]]
    train_traj = [t for t in order if t not in test_traj + val_traj]
    return train_traj, val_traj, test_traj


def build_windows(selected_traj_ids, normalized_features, normalized_target, valid_target, best_gnb):
    Xs, Ys, Yvalids, bests, starts, tids, futpos = [], [], [], [], [], [], []
    for tid in selected_traj_ids:
        idx = np.flatnonzero(trajectory_ids == tid)
        idx = idx[np.argsort(route_indices[idx])]
        n_pts = len(idx)
        max_start = n_pts - HISTORY_STEPS - FORECAST_HORIZON + 1
        for s in range(max_start):
            in_idx = idx[s: s + HISTORY_STEPS]
            tgt_idx = idx[s + HISTORY_STEPS: s + HISTORY_STEPS + FORECAST_HORIZON]
            Xs.append(normalized_features[in_idx])
            Ys.append(normalized_target[tgt_idx])
            Yvalids.append(valid_target[tgt_idx])
            bests.append(best_gnb[tgt_idx])
            starts.append(route_indices[in_idx[0]])
            tids.append(tid)
            futpos.append(positions_xyz_m[tgt_idx])
    if not Xs:
        raise RuntimeError(f"No temporal windows were created for trajectories {selected_traj_ids}.")
    return dict(
        X=np.stack(Xs).astype(np.float32),
        Y=np.stack(Ys).astype(np.float32),
        Y_valid=np.stack(Yvalids).astype(np.float32),
        future_best_gnb=np.stack(bests).astype(np.int16),
        start_route_index=np.asarray(starts, dtype=np.int32),
        trajectory_id=np.asarray(tids, dtype=np.int32),
        future_positions_xyz_m=np.stack(futpos).astype(np.float32),
    )


In [6]:
# ------------------------------------------------------------
# 5. Build and save every fold x lidar_mode combination
#    (normalization statistics are always computed from that fold's
#    training trajectories only — no leakage across folds or splits)
# ------------------------------------------------------------

manifest = {
    "source_multimodal_dataset": str(MULTIMODAL_FILE),
    "num_trajectories": int(number_of_trajectories),
    "gnb_names": gnb_names,
    "history_steps": HISTORY_STEPS,
    "forecast_horizon": FORECAST_HORIZON,
    "folds": [],
}

for lidar_mode in ["summary", "full"]:
    model_features, feature_names = build_feature_block(lidar_mode)

    for fold_index in range(number_of_trajectories):
        train_traj, val_traj, test_traj = rotating_fold_trajectories(fold_index, unique_trajectory_ids)
        train_rows = np.isin(trajectory_ids, train_traj)

        feature_mean = model_features[train_rows].mean(axis=0).astype(np.float32)
        feature_std = model_features[train_rows].std(axis=0).astype(np.float32)
        feature_std[feature_std < 1e-6] = 1.0

        target_mean = target_power_db[train_rows].mean(axis=0).astype(np.float32)
        target_std = target_power_db[train_rows].std(axis=0).astype(np.float32)
        target_std[target_std < 1e-6] = 1.0

        normalized_features = (model_features - feature_mean) / feature_std
        normalized_target = (target_power_db - target_mean) / target_std

        train_w = build_windows(train_traj, normalized_features, normalized_target, target_valid, best_gnb_index)
        val_w = build_windows(val_traj, normalized_features, normalized_target, target_valid, best_gnb_index)
        test_w = build_windows(test_traj, normalized_features, normalized_target, target_valid, best_gnb_index)

        fold_file = V2_DATA_DIR / f"fold_{fold_index}_{lidar_mode}_sequences.npz"
        np.savez_compressed(
            fold_file,
            train_X=train_w["X"], train_Y=train_w["Y"], train_Y_valid=train_w["Y_valid"],
            train_future_best_gnb=train_w["future_best_gnb"], train_trajectory_id=train_w["trajectory_id"],
            val_X=val_w["X"], val_Y=val_w["Y"], val_Y_valid=val_w["Y_valid"],
            val_future_best_gnb=val_w["future_best_gnb"], val_trajectory_id=val_w["trajectory_id"],
            test_X=test_w["X"], test_Y=test_w["Y"], test_Y_valid=test_w["Y_valid"],
            test_future_best_gnb=test_w["future_best_gnb"], test_trajectory_id=test_w["trajectory_id"],
            test_start_route_index=test_w["start_route_index"],
            test_future_positions_xyz_m=test_w["future_positions_xyz_m"],
            feature_mean=feature_mean, feature_std=feature_std,
            target_mean=target_mean, target_std=target_std,
            gnb_names=np.asarray(gnb_names, dtype="U"),
            feature_names=np.asarray(feature_names, dtype="U"),
            history_steps=np.array(HISTORY_STEPS), forecast_horizon=np.array(FORECAST_HORIZON),
        )

        manifest["folds"].append({
            "fold": fold_index, "lidar_mode": lidar_mode,
            "train_trajectories": [int(t) for t in train_traj],
            "validation_trajectory": int(val_traj[0]),
            "test_trajectory": int(test_traj[0]),
            "train_windows": int(train_w["X"].shape[0]),
            "validation_windows": int(val_w["X"].shape[0]),
            "test_windows": int(test_w["X"].shape[0]),
            "input_dimension": int(model_features.shape[1]),
            "file": str(fold_file),
        })

        print(f"[{lidar_mode:7s}] fold {fold_index} | test_traj={test_traj[0]:>2} val_traj={val_traj[0]:>2} | "
              f"windows train/val/test = {train_w['X'].shape[0]}/{val_w['X'].shape[0]}/{test_w['X'].shape[0]}")

with open(V2_RESULTS_DIR / "loto_fold_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print()
print("Saved fold manifest to:", V2_RESULTS_DIR / "loto_fold_manifest.json")
print("Total fold files written:", len(manifest["folds"]))


[summary] fold 0 | test_traj= 0 val_traj= 1 | windows train/val/test = 892/223/223
[summary] fold 1 | test_traj= 1 val_traj= 2 | windows train/val/test = 892/223/223
[summary] fold 2 | test_traj= 2 val_traj= 3 | windows train/val/test = 892/223/223
[summary] fold 3 | test_traj= 3 val_traj= 4 | windows train/val/test = 892/223/223
[summary] fold 4 | test_traj= 4 val_traj= 5 | windows train/val/test = 892/223/223
[summary] fold 5 | test_traj= 5 val_traj= 0 | windows train/val/test = 892/223/223
[full   ] fold 0 | test_traj= 0 val_traj= 1 | windows train/val/test = 892/223/223
[full   ] fold 1 | test_traj= 1 val_traj= 2 | windows train/val/test = 892/223/223
[full   ] fold 2 | test_traj= 2 val_traj= 3 | windows train/val/test = 892/223/223
[full   ] fold 3 | test_traj= 3 val_traj= 4 | windows train/val/test = 892/223/223
[full   ] fold 4 | test_traj= 4 val_traj= 5 | windows train/val/test = 892/223/223
[full   ] fold 5 | test_traj= 5 val_traj= 0 | windows train/val/test = 892/223/223

Sav